<a href="https://colab.research.google.com/github/mrunmayee3108/NeuroSolve/blob/main/Member_3_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import pandas as pd
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [ ]:
# STEP 1: LOADING BASE MODEL & ADAPTERS (OPTIMIZED)
model_id = "microsoft/Phi-3-mini-4k-instruct"
adapter_path = "./phi3-neuro-symbolic-adapter" # Member 1's output folder


In [ ]:
# Load tokenizer with left-padding fixed
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from transformers import BitsAndBytesConfig, AutoConfig # Added AutoConfig for the fix
import torch # Ensure torch is imported for bfloat16

# Load base model with SDPA speed upgrade

# Load configuration first to inspect and potentially modify
config = AutoConfig.from_pretrained(model_id, trust_remote_code=True, revision="main")

# Ensure rope_scaling is correctly handled:
# If it's a dictionary but doesn't have 'type' or if 'type' is not 'longrope',
# we set it to None to use the default RotaryEmbedding.
if hasattr(config, 'rope_scaling') and config.rope_scaling is not None:
    if not isinstance(config.rope_scaling, dict) or 'type' not in config.rope_scaling or config.rope_scaling['type'] != 'longrope':
        config.rope_scaling = None

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config, # Pass the (potentially modified) config object
    quantization_config=quantization_config, # Changed from load_in_4bit=True
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager", # Changed from "sdpa" to "eager" as per error suggestion
    revision="main" # Explicitly set revision to 'main' for consistency
)

In [ ]:
# !pip install -U transformers

In [ ]:
# Merge Member 1's brain onto the base model
import os
import zipfile

# Ensure the adapter directory exists and is unzipped
if not os.path.exists(adapter_path):
    if os.path.exists(adapter_path + ".zip"):
        print(f"Unzipping {adapter_path}.zip to {adapter_path}...")
        with zipfile.ZipFile(adapter_path + ".zip", 'r') as zip_ref:
            zip_ref.extractall(os.path.dirname(adapter_path))
    else:
        raise FileNotFoundError(f"Adapter path or zip not found: {adapter_path} or {adapter_path}.zip")

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (embed_dropout): Dropout(p=0.0, inplace=False)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnit

In [ ]:
print("--- STEP 2: DEFINING THE EXECUTION ENGINE (SANDBOX) ---")
def run_code_safely(python_code):
    """The Isolated Execution Environment."""
    # Pre-load SymPy for advanced math capabilities
    restricted_globals = {"builtins": {}}
    restricted_locals = {}

    try:
        # Prevent adversarial injection
        if any(bad_word in python_code for bad_word in ["os", "sys", "subprocess", "eval"]):
            return None, "Security Error: Malicious imports detected."

        exec(python_code, restricted_globals, restricted_locals)
        return restricted_locals.get('result', None), None
    except Exception as e:
        return None, f"{type(e).__name__}: {str(e)}"

def extract_code(text):
    """Bulletproof regex to grab the Python code."""
    # Looks specifically for code inside python ...  blocks
    match = re.search(r'python\n(.*?)\n', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

print("--- STEP 3: THE AGENTIC LOOP ---")
def solve_problem(question, max_retries=3):
    """The Neuro-Symbolic Swarm Logic."""
    history = f"### Instruction: Write Python code to solve the math problem. Store the answer in 'result'.\n### Question:\n{question}\n### Code:\n"

    for attempt in range(max_retries):
        inputs = tokenizer(history, return_tensors="pt").to("cuda")

        # SPEED UPGRADE: Greedy Decoding (temperature=0.0) for pure logic
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=False # Disable cache to resolve AttributeError: 'DynamicCache' object has no attribute 'seen_tokens'
            )

        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract the new code generated in this turn
        new_text = full_response[len(history):]
        code = extract_code(new_text)

        if not code:
            history += new_text + "\n### Error: Could not extract Python code. Please wrap code in python ... \n### Code:\n"
            continue

        # Run it in the Sandbox
        answer, error = run_code_safely(code)

        if error is None and answer is not None:
            return answer, attempt + 1 # Success!

        # If it failed, append the error and force the AI to retry
        history += new_text + f"\n### Error:\n{error}\n### Rewrite Code:\n"

    return None, max_retries

--- STEP 2: DEFINING THE EXECUTION ENGINE (SANDBOX) ---
--- STEP 3: THE AGENTIC LOOP ---


In [ ]:
test_df = pd.read_csv("unified_svamp_test.csv")

In [ ]:
test_df.shape

(700, 6)

In [ ]:
# STEP 4: EVALUATING ON SVAMP ---")
# Load your test dataset (using a tiny subset for a fast test run)
 # Remove .head(10) for the final run

correct = 0
total = 10

for index, row in test_df.iterrows():
    print(f"\nEvaluating Question {index + 1}/{total}...")
    target_answer = row['answer']

    agent_answer, attempts = solve_problem(row['question'])

    if agent_answer is not None and abs(float(agent_answer) - float(target_answer)) < 1e-4:
        print(f"Correct! (Took {attempts} attempts)")
        correct += 1
    else:
        print(f"Failed. Expected {target_answer}, Got {agent_answer}")

print("--- FINAL RESULTS ---")
print(f"Accuracy: {(correct/total)*100:.2f}%")


Evaluating Question 1/10...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 